In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm
import re
import sys
from pathlib import Path
import glob
import subprocess

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [2]:
# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing_local import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities_local import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


In [3]:
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'
local_path = "local-files"

###

EVENT_NAME = "202410_Hurricane_Milton"
product = "uavsar"

In [4]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

keys = [x.replace(f"drcs_activations/{EVENT_NAME}/{product}/", "") for x in get_all_s3_keys(s3_client, BUCKET, f"drcs_activations/{EVENT_NAME}/{product}", ".tif")] if s3_client else []

keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files


['20241011/flcorr_06206_007_24101119_quicklook_rgb.tif',
 '20241011/flcorr_24201_009_24101119_quicklook_rgb.tif',
 '20241011/peacer_19512_004_24101118_quicklook_rgb.tif',
 '20241011/tampab_15102_006_24101118_quicklook_rgb.tif',
 '20241011/tampab_33105_005_24101118_quicklook_rgb.tif',
 '20241011/tampaf_06207_010_24101120_quicklook_rgb.tif',
 '20241012/flcorr_06206_004_24101220_quicklook_class.tif',
 '20241012/flcorr_06206_004_24101220_quicklook_rgb.tif',
 '20241012/flcorr_24201_005_24101220_quicklook_class.tif',
 '20241012/flcorr_24201_005_24101220_quicklook_rgb.tif',
 '20241012/peacer_19512_001_24101219_quicklook_class.tif',
 '20241012/peacer_19512_001_24101219_quicklook_rgb.tif',
 '20241012/stjohn_17815_000_24101218_quicklook_class.tif',
 '20241012/stjohn_17815_000_24101218_quicklook_rgb.tif',
 '20241012/tampab_15102_003_24101219_quicklook_class.tif',
 '20241012/tampab_15102_003_24101219_quicklook_rgb.tif',
 '20241012/tampab_33105_002_24101219_quicklook_class.tif',
 '20241012/tampab_3

In [15]:
# Download the desired files to a local directory with a known path
local_file_dir = os.path.abspath(f"./{local_path}")
if not os.path.exists(local_file_dir):
    os.mkdir(local_file_dir)
for key in keys:
    subprocess.run([
        "aws",
        "s3",
        "cp",
        f"s3://nasa-disasters/drcs_activations/{EVENT_NAME}/{product}/{key}",
        local_file_dir], check = True)

download: s3://nasa-disasters/drcs_activations/202410_Hurricane_Milton/uavsar/20241011/flcorr_06206_007_24101119_quicklook_rgb.tif to local-files/flcorr_06206_007_24101119_quicklook_rgb.tif
download: s3://nasa-disasters/drcs_activations/202410_Hurricane_Milton/uavsar/20241011/flcorr_24201_009_24101119_quicklook_rgb.tif to local-files/flcorr_24201_009_24101119_quicklook_rgb.tif
download: s3://nasa-disasters/drcs_activations/202410_Hurricane_Milton/uavsar/20241011/peacer_19512_004_24101118_quicklook_rgb.tif to local-files/peacer_19512_004_24101118_quicklook_rgb.tif
download: s3://nasa-disasters/drcs_activations/202410_Hurricane_Milton/uavsar/20241011/tampab_15102_006_24101118_quicklook_rgb.tif to local-files/tampab_15102_006_24101118_quicklook_rgb.tif
download: s3://nasa-disasters/drcs_activations/202410_Hurricane_Milton/uavsar/20241011/tampab_33105_005_24101118_quicklook_rgb.tif to local-files/tampab_33105_005_24101118_quicklook_rgb.tif
download: s3://nasa-disasters/drcs_activations/202

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()


# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

In [6]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [7]:
def make_regex_dict(keys, regexes, products):
    ret = {}
    for i in range(len(products)):
        matches = []
        for key in keys:
            filename = key.split("/")[-1]
            match = re.search(regexes[i], filename)
            if match is not None:
                matches.append(key)
        if matches != []:
            if products[i] in ret.keys():
                ret[products[i]] = ret[products[i]] + matches
            else:
                ret[products[i]] = matches
    return ret

In [8]:
def create_cog_filename(filename, event):
    if re.search(r".*quicklook_rgb.tif", filename) is not None:
        sname = filename.split("/")[-1].replace(".tif", "").split("_")
        date = datetime.strptime(f"20{sname[3]}", "%Y%m%d%H")
        new_dt_format = date.strftime("%Y-%m-%dT%H:%M:%SZ")
        cog_filename = f"{event}_{sname[1]}_{sname[2]}_{sname[4]}_{sname[5]}_{sname[0]}_{new_dt_format}.tif"

    elif re.search(r".*_quicklook_class.tif", filename) is not None:
        sname = filename.split("/")[-1].replace(".tif", "").split("_")
        date = datetime.strptime(f"20{sname[3]}", "%Y%m%d%H")
        new_dt_format = date.strftime("%Y-%m-%dT%H:%M:%SZ")
        cog_filename = f"{event}_{sname[1]}_{sname[2]}_{sname[4]}_{sname[5]}_{sname[0]}_{new_dt_format}.tif"

    elif re.search(r".*uavsar.*_rgb.tif", filename) is not None:
        sname = filename.split("/")[-1].replace(".tif", "").split("_")
        date = datetime.strptime(sname[3], "%Y%m%d")
        new_dt_format = date.strftime("%Y-%m-%d_day")
        cog_filename = f"{event}_{sname[0]}_{sname[1]}_{sname[2]}_{sname[4]}_{new_dt_format}.tif"

    elif re.search(r".*uavsar.*_class.tif", filename) is not None:
        sname = filename.split("/")[-1].replace(".tif", "").split("_")
        date = datetime.strptime(sname[3], "%Y%m%d")
        new_dt_format = date.strftime("%Y-%m-%d_day")
        cog_filename = f"{event}_{sname[0]}_{sname[1]}_{sname[2]}_{sname[4]}_{sname[5]}_{new_dt_format}.tif"

    else:
        print(f"{filename} not caught by regexes!")
        return None
    
    return cog_filename

In [9]:
local_keys = [x for x in glob.glob(f"{local_path}/*") if x.endswith(".tif")]

local_keys

['local-files/peacer_19512_006_24101419_quicklook_class.tif',
 'local-files/flcorr_06206_007_24101119_quicklook_rgb.tif',
 'local-files/stjohn_17815_005_24101419_quicklook_class.tif',
 'local-files/flcorr_24201_009_24101119_quicklook_rgb.tif',
 'local-files/uavsar_flight_24075_20241011_UNet_class.tif',
 'local-files/peacer_19512_004_24101118_quicklook_rgb.tif',
 'local-files/stjohn_17815_005_24101419_quicklook_rgb.tif',
 'local-files/tampab_15102_006_24101118_quicklook_rgb.tif',
 'local-files/stjohn_35821_004_24101418_quicklook_class.tif',
 'local-files/tampab_33105_005_24101118_quicklook_rgb.tif',
 'local-files/stjohn_35821_004_24101418_quicklook_rgb.tif',
 'local-files/tampaf_06207_010_24101120_quicklook_rgb.tif',
 'local-files/stjohn_35821_008_24101320_quicklook_class.tif',
 'local-files/flcorr_06206_004_24101220_quicklook_class.tif',
 'local-files/tampab_33105_007_24101419_quicklook_class.tif',
 'local-files/flcorr_06206_004_24101220_quicklook_rgb.tif',
 'local-files/stjohn_17815_0

In [10]:
reg_keys = make_regex_dict(local_keys, [r".*quicklook_rgb.tif", r".*_quicklook_class.tif", r".*uavsar.*_rgb.tif", r".*uavsar.*_class.tif"], ["RGB", "class", "RGB", "class"])

In [11]:
print(reg_keys)
for k, v in reg_keys.items():
    for filename in v:
        print(create_cog_filename(filename, EVENT_NAME))

{'RGB': ['local-files/flcorr_06206_007_24101119_quicklook_rgb.tif', 'local-files/flcorr_24201_009_24101119_quicklook_rgb.tif', 'local-files/peacer_19512_004_24101118_quicklook_rgb.tif', 'local-files/stjohn_17815_005_24101419_quicklook_rgb.tif', 'local-files/tampab_15102_006_24101118_quicklook_rgb.tif', 'local-files/tampab_33105_005_24101118_quicklook_rgb.tif', 'local-files/stjohn_35821_004_24101418_quicklook_rgb.tif', 'local-files/tampaf_06207_010_24101120_quicklook_rgb.tif', 'local-files/flcorr_06206_004_24101220_quicklook_rgb.tif', 'local-files/stjohn_17815_009_24101321_quicklook_rgb.tif', 'local-files/tampab_33105_007_24101419_quicklook_rgb.tif', 'local-files/flcorr_24201_005_24101220_quicklook_rgb.tif', 'local-files/stjohn_35821_008_24101320_quicklook_rgb.tif', 'local-files/peacer_19512_001_24101219_quicklook_rgb.tif', 'local-files/tampaf_06207_000_24101416_quicklook_rgb.tif', 'local-files/stjohn_17815_000_24101218_quicklook_rgb.tif', 'local-files/tampab_15102_002_24101317_quickloo

In [12]:
def simple_process_files(file_list, rename_func, target_dir, event):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    print("Testing filenams:")
    for filename in file_list:
        print(f"  {rename_func(filename, event)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        #"raw_data_bucket": BUCKET,
        #"raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{event}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(local_download_path, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            local_download_path, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=file_list,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=event,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

In [13]:
if not os.path.exists(os.path.abspath("./output")):
    os.mkdir(os.path.abspath("./output"))
if not os.path.exists(os.path.abspath("./reproj")):
    os.mkdir(os.path.abspath("./reproj"))
for k, v in reg_keys.items():
    results = simple_process_files(file_list = v[-4:], rename_func = create_cog_filename, target_dir = f"UAVSAR/{k}", event = EVENT_NAME)

Testing filenams:
  202410_Hurricane_Milton_uavsar_flight_24075_rgb_2024-10-11_day.tif
  202410_Hurricane_Milton_uavsar_flight_24076_rgb_2024-10-12_day.tif
  202410_Hurricane_Milton_uavsar_flight_24077_rgb_2024-10-13_day.tif
  202410_Hurricane_Milton_uavsar_flight_24078_rgb_2024-10-14_day.tif
Configuration loaded:
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/UAVSAR/RGB

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202410_Hurricane_Milton

[1/4] Processing: local-files/uavsar_flight_24075_20241011_rgb.tif
   Output filename: 202410_Hurricane_Milton_uavsar_flight_24075_rgb_2024-10-11_day.tif
   [CACHE HIT] Using local file: local-files/uavsar_flight_24075_20241011_rgb.tif
   [MEMORY] Initial: 294.6 MB
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=3, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 40.0% (from distributed samples

/srv/conda/envs/notebook/lib/python3.12/site-packages/rio_cogeo/cogeo.py:226: NodataAlphaMaskWarning: Input dataset has both a nodata value and internal alpha/mask band. Nodata value will be prioritized.
  warnings.warn(
Reading input: /tmp/tmplw70ixdz_temp.tif

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp59j435s4.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/UAVSAR/RGB/202410_Hurricane_Milton_uavsar_flight_24075_rgb_2024-10-11_day.tif
   [MEMORY] Final: 3023.5 MB (Change: +2728.8 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Hurricane_Milton_uavsar_flight_24075_rgb_2024-10-11_day.tif

[2/4] Processing: local-files/uavsar_flight_24076_20241012_rgb.tif
   Output filename: 202410_Hurricane_Milton_uavsar_flight_24076_rgb_2024-10-12_day.tif
   [CACHE HIT] Using local file: local-files/uavsar_flight_24076_20241012_rgb.tif
   [MEMORY] Initial: 3023.5 MB
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated da

/srv/conda/envs/notebook/lib/python3.12/site-packages/rio_cogeo/cogeo.py:226: NodataAlphaMaskWarning: Input dataset has both a nodata value and internal alpha/mask band. Nodata value will be prioritized.
  warnings.warn(
Reading input: /tmp/tmp0urue790_temp.tif

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpc2l_e1cd.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/UAVSAR/RGB/202410_Hurricane_Milton_uavsar_flight_24076_rgb_2024-10-12_day.tif
   [MEMORY] Final: 2140.4 MB (Change: -883.1 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Hurricane_Milton_uavsar_flight_24076_rgb_2024-10-12_day.tif

[3/4] Processing: local-files/uavsar_flight_24077_20241013_rgb.tif
   Output filename: 202410_Hurricane_Milton_uavsar_flight_24077_rgb_2024-10-13_day.tif
   [CACHE HIT] Using local file: local-files/uavsar_flight_24077_20241013_rgb.tif
   [MEMORY] Initial: 2139.4 MB
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=3, max=255, center sample non-zero=1000000/1000000
            Estimated dat

...

   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/UAVSAR/class/202410_Hurricane_Milton_uavsar_flight_24077_UNet_class_2024-10-13_day.tif
   [MEMORY] Final: 2372.3 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Hurricane_Milton_uavsar_flight_24077_UNet_class_2024-10-13_day.tif

[4/4] Processing: local-files/uavsar_flight_24078_20241014_UNet_class.tif
   Output filename: 202410_Hurricane_Milton_uavsar_flight_24078_UNet_class_2024-10-14_day.tif
   [CACHE HIT] Using local file: local-files/uavsar_flight_24078_20241014_UNet_class.tif
   [MEMORY] Initial: 2372.3 MB
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=255, center sample non-zero=125940

/srv/conda/envs/notebook/lib/python3.12/site-packages/rio_cogeo/cogeo.py:226: NodataAlphaMaskWarning: Input dataset has both a nodata value and internal alpha/mask band. Nodata value will be prioritized.
  warnings.warn(
Reading input: /tmp/tmp96bulxvt_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpv8kqde7y.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/UAVSAR/class/202410_Hurricane_Milton_uavsar_flight_24078_UNet_class_2024-10-14_day.tif
   [MEMORY] Final: 2372.3 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Hurricane_Milton_uavsar_flight_24078_UNet_class_2024-10-14_day.tif

✅ Batch processing complete: 4 files processed
📁 COGs saved locally to: output/202410_Hurricane_Milton

📊 BATCH PROCESSING SUMMARY
Total files processed: 4
Successful: 4
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-30T03:17:11.283826


In [14]:
subprocess.run(["rm", "-r", f"{local_path}"], check = True)
subprocess.run(["rm", "-r", os.path.abspath("./output")], check = True)
subprocess.run(["rm", "-r", os.path.abspath("./reproj")], check = True)

CompletedProcess(args=['rm', '-r', '/home/jovyan/conversion_scripts/convert-files-and-move/2024/reproj'], returncode=0)